# Task 3 (Medium) – Recommendation System Analysis
**Internship:** Auspify Technologies – Data Science Program  
**Intern:** Shabbir Moomin (Colombo, Sri Lanka)  
**Dataset:** `Netflix_Cleaned.csv` (Task 1 output – same records as the provided `Dataset_1.csv`)

**Goal:** build a basic *content-based* recommender: "if you liked title X, here are similar titles."

**Workflow**
1. Extract relevant content features
2. Perform text preprocessing
3. Calculate content similarity
4. Generate recommendations for selected titles
5. Evaluate recommendation quality

**A note on the data:** there is no plot/description column and no user-rating data, so this is *content-based filtering* built from metadata
(genres, director, country, rating, type, title words). Collaborative filtering (user-item) is not possible here.

In [62]:
import re
import difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel

sns.set_theme(style='whitegrid')
RED, GREY = '#E50914', '#564d4d'
np.random.seed(42)  # reproducible random sampling (42, obviously)

df = pd.read_csv('Netflix_Cleaned.csv', parse_dates=['date_added'])
print(df.shape)
df[['title', 'type', 'director', 'country', 'rating', 'listed_in']].head()

(8786, 34)


,title,type,director,country,rating,listed_in
0,Dick Johnson Is Dead,Movie,Kirsten Johnson,United States,PG-13,Documentaries
1,Ganglands,TV Show,Julien Leclercq,France,TV-MA,"Crime TV Shows, International TV Shows, TV Act..."
2,Midnight Mass,TV Show,Mike Flanagan,United States,TV-MA,"TV Dramas, TV Horror, TV Mysteries"
3,Confessions of an Invisible Girl,Movie,Bruno Garotti,Brazil,TV-PG,"Children & Family Movies, Comedies"
4,Sankofa,Movie,Haile Gerima,United States,TV-MA,"Dramas, Independent Movies, International Movies"


## Step 1 – Extract relevant content features

Which columns actually say something about *what a title is like*?

| Column | Use? | Reason |
|---|---|---|
| `listed_in` (genres) | ✅ strongest | directly describes content |
| `director` | ✅ | fans often like the same director; also links franchises |
| `country` | ✅ | similar regional cinema |
| `audience` / `rating` | ✅ | don't recommend a kids' cartoon after a TV-MA thriller |
| `type` | ✅ | movie vs TV show |
| `title` | ✅ (light) | words like *Naruto*, *Panda*, *Heist* connect sequels / franchises |
| `show_id`, `date_added`, `duration` | ❌ | administrative / not content descriptors |

In [ ]:
# how many titles have usable (non-Unknown) director / country info? matters for how much weight we give them
print(f"director known: {(df['director'] != 'Unknown').mean():.1%}")
print(f"country known : {(df['country'] != 'Unknown').mean():.1%}")
print('distinct genre labels:', df['listed_in'].str.split(',').explode().str.strip().nunique())

## Step 2 – Text preprocessing
Metadata isn't a sentence, so preprocessing is about making tokens **comparable**:

- lower-casing, removing punctuation
- **multi-word names become one token** (`"Raúl Campos"` → `raulcampos`) – otherwise "Campos" would match any other Campos
- genre labels are normalised so Movie and TV genres line up: `"TV Dramas"` and `"Dramas"` → `dramas`; `"Crime TV Shows"` → `crime`
- placeholders (`Unknown`) are dropped, they carry no information
- title words: stop-words removed, very short tokens removed

In [ ]:
STOP = {'the','a','an','of','and','in','on','to','for','with','my','me','you','your','is','at','by','from','vs',
        'part','season','movie','film','story','new','life','one','two'}   # small hand-made stop-word list

def clean_token(s):
    # lower-case, strip accents-ish punctuation, remove spaces -> single token
    s = re.sub(r'[^a-z0-9]+', '', s.lower())
    return s

def norm_genre(g):
    # 'TV Dramas' -> 'dramas', 'Crime TV Shows' -> 'crime', 'Anime Features' -> 'anime'
    g = re.sub(r'\b(tv|movies|shows|series|features)\b', ' ', g.lower())
    return clean_token(g)

def title_tokens(t):
    words = re.findall(r'[a-z0-9]+', t.lower())
    return [w for w in words if w not in STOP and len(w) > 2 and not w.isdigit()]

def build_soup(row, w_genre=3, w_dir=2, w_country=1, w_aud=1, w_type=1, w_title=1):
    # every feature becomes tokens, repeated 'weight' times -> higher weight = more influence on similarity
    genres    = [norm_genre(g) for g in row['listed_in'].split(',')]
    genres    = [g for g in genres if g]
    directors = [] if row['director'] == 'Unknown' else [clean_token(d) for d in row['director'].split(',')]
    country   = [] if row['country']  == 'Unknown' else [clean_token(c) for c in row['country'].split(',')]
    aud       = ['aud_' + clean_token(str(row['audience']))]
    typ       = ['type_' + clean_token(row['type'])]
    title     = title_tokens(row['title'])
    toks = (genres * w_genre + directors * w_dir + country * w_country +
            aud * w_aud + typ * w_type + title * w_title)
    return ' '.join(toks)

df['soup'] = df.apply(build_soup, axis=1)
df[['title', 'soup']].sample(4, random_state=1).to_dict('records')

## Step 3 – Calculate content similarity

**TF-IDF + cosine similarity.**  
TF-IDF down-weights tokens that appear everywhere (e.g. `dramas`, `type_movie`) and up-weights rare, informative ones (a specific director or `anime`). Cosine similarity then measures how much two titles' token profiles point in the same direction (0 = nothing in common, 1 = identical).

In [ ]:
# analyzer=str.split -> tokens are already clean, so just split on spaces
tfidf = TfidfVectorizer(analyzer=str.split)
X = tfidf.fit_transform(df['soup'])
print('TF-IDF matrix:', X.shape, '| non-zeros per row ~', round(X.nnz / X.shape[0], 1))

# NOTE: a full 8786x8786 similarity matrix is ~300-600 MB, so we compute similarities per query instead
# (still instant). The evaluation section does it in batches.
indices = pd.Series(df.index, index=df['title'].str.lower()).groupby(level=0).first()

In [ ]:
def find_title(query):
    # exact match first, then fuzzy match (typos / partial names), returns row index
    q = query.lower().strip()
    if q in indices.index:
        return indices[q]
    close = difflib.get_close_matches(q, indices.index, n=1, cutoff=0.6)
    if not close:
        raise ValueError(f"No title like '{query}' found")
    print(f"(no exact match - using closest title: '{close[0]}')")
    return indices[close[0]]

## Step 4 – Generate recommendations for selected titles

In [ ]:
def recommend(title, n=10, same_type=False, matrix=X):
    idx = find_title(title)
    sims = linear_kernel(matrix[idx], matrix).ravel()     # cosine sim (TF-IDF rows are L2-normalised)
    sims[idx] = -1                                         # don't recommend the title itself
    if same_type:
        sims[(df['type'] != df.loc[idx, 'type']).values] = -1
    top = np.argsort(-sims)[:n]
    out = df.iloc[top][['title', 'type', 'listed_in', 'director', 'country', 'rating']].copy()
    out.insert(1, 'similarity', sims[top].round(3))
    print(f"Because you watched: {df.loc[idx, 'title']}  [{df.loc[idx, 'type']} | {df.loc[idx, 'listed_in']}]")
    return out.reset_index(drop=True)

recommend('Stranger Things')

In [ ]:
recommend('Kung Fu Panda', n=8)

In [ ]:
recommend('Peaky Blinders', n=8)

In [ ]:
recommend('Zombieland', n=8)

In [ ]:
# cross-type vs same-type: Narcos-style crime docuseries
recommend('Black Mirror', n=8, same_type=True)

In [ ]:
# typo test - fuzzy matching should rescue this
recommend('stranger thinks', n=5)

In [ ]:
# visualise similarity scores for one query
r = recommend('Peaky Blinders', n=10)
plt.figure(figsize=(9, 4.5))
sns.barplot(data=r, x='similarity', y='title', color=RED)
plt.title('Top 10 recommendations for "Peaky Blinders" (cosine similarity)')
plt.ylabel(''); plt.show()

## Step 5 – Evaluate recommendation quality

There are **no user ratings**, so we can't compute accuracy / RMSE / hit-rate. Instead we use *offline proxy metrics* on a random sample of 1,000 titles (top-10 recommendations each):

| Metric | Meaning | Good = |
|---|---|---|
| **Genre overlap (Jaccard)** | genre-set similarity between a title and its recommendations | higher |
| **Shares ≥1 genre** | % of recommendations with at least one genre in common | higher |
| **Same audience group** | recommendation has same audience group (kids/teens/adults…) | higher |
| **Catalogue coverage** | % of titles that appear in anyone's top-10 (are we recommending the same few titles?) | higher |
| **Intra-list diversity** | 1 − average similarity among the 10 recommendations | moderate |

Because genres are *inside* the model, genre metrics alone are a bit circular. So I also run an **ablation: a model that never sees genres** (director, country, audience, type, title words only) and check whether it still recovers genre-relevant titles better than random. A **random recommender** is the baseline.

In [ ]:
# multi-hot genre matrix (normalised genres) used only for scoring
genre_sets = df['listed_in'].apply(lambda s: {g for g in (norm_genre(x) for x in s.split(',')) if g})
sample_idx = np.random.choice(len(df), 1000, replace=False)

def jaccard(a, b):
    return len(a & b) / len(a | b) if (a or b) else 0

def evaluate(matrix, name, k=10, rand=False):
    S = sample_idx
    if rand:
        recs = np.array([np.random.choice(np.delete(np.arange(len(df)), i), k, replace=False) for i in S])
    else:
        sims = (matrix[S] @ matrix.T).toarray()
        sims[np.arange(len(S)), S] = -1                  # exclude self
        recs = np.argpartition(-sims, k, axis=1)[:, :k]  # top-k (unordered, fine for these metrics)
    jac, share, aud = [], [], []
    for i, r in zip(S, recs):
        jac.append(np.mean([jaccard(genre_sets[i], genre_sets[j]) for j in r]))
        share.append(np.mean([len(genre_sets[i] & genre_sets[j]) > 0 for j in r]))
        aud.append(np.mean((df['audience'].values[r] == df['audience'].values[i])))
    # diversity needs pairwise sims among the recs
    if rand or matrix is None:
        div = np.nan
    else:
        d = []
        for r in recs[:300]:
            sub = cosine_similarity(matrix[r]); d.append(1 - (sub.sum() - k) / (k * (k - 1)))
        div = np.mean(d)
    # coverage over ALL titles, in batches
    seen = set()
    if not rand:
        for start in range(0, len(df), 1000):
            b = (matrix[start:start + 1000] @ matrix.T).toarray()
            b[np.arange(b.shape[0]), np.arange(start, start + b.shape[0])] = -1
            seen.update(np.argpartition(-b, k, axis=1)[:, :k].ravel().tolist())
    cov = len(seen) / len(df) if not rand else np.nan
    return dict(model=name, genre_jaccard=np.mean(jac), shares_genre=np.mean(share),
                same_audience=np.mean(aud), coverage=cov, diversity=div)

In [ ]:
# model variants for comparison
def make_matrix(**weights):
    soup = df.apply(lambda r: build_soup(r, **weights), axis=1)
    return TfidfVectorizer(analyzer=str.split).fit_transform(soup), soup

X_full = X
X_nogenre, _ = make_matrix(w_genre=0)                       # ablation: no genres at all
X_genre, _   = make_matrix(w_dir=0, w_country=0, w_aud=0, w_type=0, w_title=0)   # genres only
soup_full = df['soup']
X_count = CountVectorizer(analyzer=str.split).fit_transform(soup_full).astype(float)  # raw counts instead of TF-IDF
from sklearn.preprocessing import normalize
X_count = normalize(X_count)                                # L2-normalise so dot product = cosine

results = pd.DataFrame([
    evaluate(None, 'Random baseline', rand=True),
    evaluate(X_count, 'Count vectors (full features)'),
    evaluate(X_genre, 'TF-IDF – genres only'),
    evaluate(X_nogenre, 'TF-IDF – NO genres (ablation)'),
    evaluate(X_full, 'TF-IDF – full model (initial weights)'),
]).set_index('model')
results.round(3)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
for a, col, ttl in zip(ax, ['genre_jaccard', 'shares_genre', 'coverage'],
                       ['Genre overlap (Jaccard)', '% recs sharing >=1 genre', 'Catalogue coverage']):
    results[col].dropna().sort_values().plot(kind='barh', ax=a, color=[GREY if 'Random' in i else RED for i in results[col].dropna().sort_values().index])
    a.set_title(ttl); a.set_ylabel('')
plt.tight_layout(); plt.show()

### Tuning the feature weights
The proxy metrics show the *initial* weights (genre ×3, director ×2, audience ×1) are not ideal: the high-IDF director/title tokens pull recommendations away from the right audience group (only ~60% match). I try a few weight combinations and pick the one with the best **equal-weighted average of the four metrics** (genre overlap, same audience, coverage, diversity) so that one metric can't be gamed at the expense of the others – e.g. a genres-only model has near-perfect genre scores but its recommendations are all clones of each other (diversity ≈ 0.01).

In [ ]:
grid = {
    'A initial (3,2,1,1,1)': dict(w_genre=3, w_dir=2, w_country=1, w_aud=1, w_type=1, w_title=1),
    'B (3,1,1,2,1)':        dict(w_genre=3, w_dir=1, w_country=1, w_aud=2, w_type=1, w_title=1),
    'C (4,1,1,2,1)':        dict(w_genre=4, w_dir=1, w_country=1, w_aud=2, w_type=1, w_title=1),
    'D (4,2,1,3,1)':        dict(w_genre=4, w_dir=2, w_country=1, w_aud=3, w_type=1, w_title=1),
    'G (4,1,1,3,1)':        dict(w_genre=4, w_dir=1, w_country=1, w_aud=3, w_type=1, w_title=1),
}   # label = (genre, director, country, audience, title) weights

rows, mats = [], {}
for name, w in grid.items():
    M, _ = make_matrix(**w)
    mats[name] = M
    rows.append(evaluate(M, name))
tuning = pd.DataFrame(rows).set_index('model')
tuning['score'] = tuning[['genre_jaccard', 'same_audience', 'coverage', 'diversity']].mean(axis=1)
best_name = tuning['score'].idxmax()
print('Best configuration:', best_name)
tuning.round(3)

In [ ]:
ax = tuning['score'].sort_values().plot(kind='barh', color=[RED if i == best_name else GREY for i in tuning['score'].sort_values().index])
ax.set_title('Weight tuning - combined quality score'); ax.set_ylabel(''); ax.set_xlabel('mean of 4 metrics')
plt.show()

In [ ]:
# final model = best configuration from the tuning grid
X_final = mats[best_name]
print('Final model:', best_name, '| matrix', X_final.shape)

In [ ]:
# same example titles again with the tuned model -> compare with the earlier output
recommend('Stranger Things', n=8, matrix=X_final)

In [ ]:
recommend('Zombieland', n=8, matrix=X_final)

In [ ]:
recommend('Kung Fu Panda', n=8, matrix=X_final)

In [ ]:
# manual sanity check: do franchise / sequel titles end up together?  (nice qualitative test)
tests = {'Kung Fu Panda': 'panda', 'Naruto Shippuden the Movie: Blood Prison': 'naruto', 'My Little Pony: A New Generation': 'pony'}
for t, kw in tests.items():
    r = recommend(t, n=10, matrix=X_final)
    print(f"  -> recommendations containing '{kw}':", int(r['title'].str.lower().str.contains(kw).sum()), '/ 10\n')

In [ ]:
# where does the model struggle? titles with little metadata (Unknown country = 287 titles, no director either in many cases)
weak = df[(df['country'] == 'Unknown') & (df['director'] == 'Unknown')]
print('Titles with Unknown director AND country:', len(weak))
weak = weak if len(weak) else df[df['country'] == 'Unknown']
recommend(weak.iloc[0]['title'], n=5, matrix=X_final)

## Summary of results

**Approach:** content-based recommender - metadata (genres, director, country, audience group, type, title words) → cleaned tokens → TF-IDF vectors → cosine similarity → top-N most similar titles.

**Evaluation (1,000 random titles, top-10 each)**

| Model | Genre overlap | Shares ≥1 genre | Same audience | Coverage | Diversity |
|---|---|---|---|---|---|
| Random baseline | 0.14 | 39% | 33% | – | – |
| No-genre ablation | 0.27 | 59% | 58% | 84% | 0.74 |
| Initial weights (3,2,1,1,1) | 0.82 | 97% | 60% | 70% | 0.50 |
| **Final tuned model (3,1,1,2,1)** | **0.88** | **99%** | **82%** | **77%** | **0.43** |

**Key takeaways**
- The final model is far better than random: genre overlap is ~6× the random baseline, and 99% of recommendations share at least one genre with the query title.
- Even **without genres**, director + country + audience + title words roughly double the random genre overlap (0.27 vs 0.14), so those features carry real signal.
- **Tuning mattered:** raising the audience weight and lowering the director weight lifted same-audience matches from 60% → 82% and catalogue coverage from 70% → 77%, at a small cost in diversity (0.50 → 0.43). Genre-only models score highest on genre metrics but return near-identical clones (diversity ≈ 0.01), so they are not truly better.
- Qualitatively the results look sensible: *Stranger Things* → *The OA*, *Helix*, *Chilling Adventures of Sabrina*; *Kung Fu Panda* → its sequels and specials first; *Zombieland* → *The Babysitter*, *iZombie*, *Z Nation*.

**Limitations (worth stating in the report)**
- No plot descriptions → the model can't see actual story content; two "TV Dramas" can be very different shows.
- No user ratings or watch history → quality is measured with *proxy metrics* only. Genre-based metrics are partly circular because genres are an input; the ablation and random baseline are there to show the result isn't just an artefact.
- Title words occasionally create false matches (e.g. *My Little Pony: A New Generation* → *The Mars Generation*). Lowering the title weight or removing generic words would reduce this.
- ~29% of directors and ~3% of countries are Unknown, so those titles rely mostly on genre and audience.
- Popularity is not modelled - a niche title and a global hit are treated equally.

**Possible improvements:** add plot descriptions (TF-IDF / sentence embeddings), add user ratings for hybrid or collaborative filtering, use maximal-marginal-relevance re-ranking for more diverse lists, and run an online A/B test with real viewers.
